# 01 — Tensor & Autograd di PyTorch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/01-deep-learning/01_tensor_autograd.ipynb)

Di notebook ini kita belajar fondasi PyTorch:

1. **Tensor**: struktur data utama (mirip `numpy.ndarray`, tapi bisa jalan di GPU).
2. **Operasi dasar**: aritmetika, perkalian matriks, broadcasting, reshape, indexing.
3. **Device**: memindahkan tensor ke GPU jika tersedia.
4. **Autograd**: PyTorch menghitung turunan (gradien) secara otomatis.
5. **Gradient descent manual**: melatih regresi linear sederhana *tanpa* `nn.Module` maupun `optimizer`, supaya terlihat apa yang sebenarnya terjadi saat training.

> 💡 Di Google Colab, PyTorch sudah terpasang. Untuk memakai GPU: **Runtime → Change runtime type → GPU**. Notebook ini juga tetap jalan di CPU.

## 0. Setup

Import library dan set *random seed* agar hasil angka acak bisa direproduksi (hasilnya sama setiap kali dijalankan).

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# Seed supaya angka acak konsisten di setiap run
torch.manual_seed(42)

print("Versi PyTorch:", torch.__version__)

## 1. Membuat Tensor

Tensor adalah array multi-dimensi:
- skalar = tensor 0 dimensi, vektor = 1 dimensi, matriks = 2 dimensi, gambar berwarna = 3 dimensi (C, H, W), batch gambar = 4 dimensi (N, C, H, W).

Tiga atribut penting yang selalu perlu dicek: **`shape`** (ukuran), **`dtype`** (tipe data), dan **`device`** (CPU/GPU).

In [ ]:
# Dari list Python
a = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(a)
print("shape :", a.shape)   # 2 baris, 3 kolom
print("dtype :", a.dtype)   # integer 64-bit karena input berupa int
print("device:", a.device)  # default-nya CPU

Beberapa fungsi untuk membuat tensor dengan pola tertentu. Perhatikan: bobot neural network biasanya bertipe `float32`, jadi kita sering memakai tensor float.

In [ ]:
nol = torch.zeros(2, 3)          # semua 0
satu = torch.ones(2, 3)          # semua 1
acak = torch.rand(2, 3)          # acak uniform [0, 1)
normal = torch.randn(2, 3)       # acak distribusi normal (mean 0, std 1)
urut = torch.arange(0, 10, 2)    # 0, 2, 4, 6, 8
jarak = torch.linspace(0, 1, 5)  # 5 titik rata dari 0 sampai 1

for nama, t in [("zeros", nol), ("ones", satu), ("rand", acak),
                ("randn", normal), ("arange", urut), ("linspace", jarak)]:
    print(f"{nama:8s} dtype={str(t.dtype):14s} {t.tolist()}")

### Konversi dengan NumPy

`torch.from_numpy()` dan `.numpy()` **berbagi memori** (untuk tensor di CPU): mengubah salah satu akan mengubah yang lain. Gunakan `.clone()` jika ingin salinan terpisah.

In [ ]:
arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(arr)   # berbagi memori dengan arr
arr[0] = 100.0
print("tensor setelah arr diubah:", t)  # ikut berubah!

kembali = t.numpy()         # tensor -> numpy
print("tipe:", type(kembali))

# Ubah dtype: float64 (default numpy) -> float32 (default PyTorch)
print("dtype float32:", t.float().dtype)

## 2. Operasi Dasar

### Aritmetika element-wise dan perkalian matriks

- `+ - * /` bekerja **per elemen** (ukuran harus sama atau bisa di-*broadcast*).
- `@` (atau `torch.matmul`) adalah **perkalian matriks**: `(m, n) @ (n, p) -> (m, p)`. Ini operasi paling penting di neural network.

In [ ]:
x = torch.tensor([[1., 2.], [3., 4.]])
y = torch.tensor([[10., 20.], [30., 40.]])

print("x + y =\n", x + y)
print("x * y (per elemen) =\n", x * y)
print("x @ y (perkalian matriks) =\n", x @ y)

### Broadcasting

Jika ukuran tidak sama, PyTorch otomatis "memperluas" dimensi berukuran 1 agar cocok. Contoh umum: menambahkan **bias** (vektor) ke setiap baris output (matriks).

In [ ]:
output = torch.ones(4, 3)          # misal: 4 sampel, 3 neuron
bias = torch.tensor([0.1, 0.2, 0.3])  # shape (3,)

hasil = output + bias  # bias ditambahkan ke setiap baris
print(hasil)
print("shape:", hasil.shape)

### Reshape, indexing, dan reduksi

- `reshape` / `view`: mengubah bentuk tanpa mengubah isi. `-1` artinya "hitung otomatis".
- Indexing mirip NumPy.
- Reduksi (`sum`, `mean`, `max`, `argmax`) bisa dilakukan per dimensi dengan argumen `dim`.

In [ ]:
t = torch.arange(12)
print("asli      :", t.shape)
m = t.reshape(3, 4)
print("reshape   :\n", m)
print("flatten   :", m.reshape(-1).shape)

print("baris 0   :", m[0])
print("kolom 1   :", m[:, 1])
print("elemen    :", m[2, 3].item())  # .item() -> angka Python

mf = m.float()
print("sum total :", mf.sum().item())
print("mean dim=0 (per kolom):", mf.mean(dim=0))
print("argmax dim=1 (per baris):", mf.argmax(dim=1))  # dipakai untuk ambil kelas prediksi

## 3. Memindahkan Tensor ke GPU

Pola standar: tentukan `device` sekali di awal, lalu pindahkan tensor (dan nanti model) dengan `.to(device)`.

⚠️ Semua tensor dalam satu operasi harus berada di **device yang sama**. Menjumlahkan tensor CPU dengan tensor GPU akan error.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device yang dipakai:", device)

a = torch.randn(3, 3)
a_dev = a.to(device)       # salinan di device tujuan
print("a     di:", a.device)
print("a_dev di:", a_dev.device)

# Operasi dilakukan di device tempat tensor berada
b_dev = a_dev @ a_dev

# Untuk plotting / numpy, pindahkan kembali ke CPU
print(b_dev.cpu().numpy().round(2))

## 4. Autograd: Turunan Otomatis

Jika tensor dibuat dengan `requires_grad=True`, PyTorch mencatat setiap operasi yang melibatkannya (membentuk **computational graph**). Memanggil `.backward()` pada hasil akhir (skalar) akan menghitung turunan terhadap tensor tersebut dan menyimpannya di atribut `.grad`.

Contoh: $y = x^2 + 3x$, maka $\frac{dy}{dx} = 2x + 3$. Di $x = 2$, turunannya $= 7$.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x

y.backward()          # hitung dy/dx
print("y       =", y.item())
print("dy/dx   =", x.grad.item())
print("manual  =", 2 * 2.0 + 3)

### Gradien untuk banyak variabel

Contoh lebih mirip neural network: $L = \text{mean}((w \cdot x + b - y)^2)$. Autograd menghitung $\partial L / \partial w$ dan $\partial L / \partial b$ sekaligus.

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)
xs = torch.tensor([1.0, 2.0, 3.0])
ys = torch.tensor([3.0, 5.0, 7.0])  # sebenarnya y = 2x + 1

loss = ((w * xs + b - ys) ** 2).mean()
loss.backward()
print("loss  :", loss.item())
print("dL/dw :", w.grad.item())
print("dL/db :", b.grad.item())

# Cek manual: dL/dw = mean(2 * (w*x + b - y) * x), dL/db = mean(2 * (w*x + b - y))
err = (w.detach() * xs + b.detach() - ys)
print("manual dL/dw:", (2 * err * xs).mean().item())
print("manual dL/db:", (2 * err).mean().item())

### Tiga hal penting tentang autograd

1. **Gradien diakumulasi (ditambahkan)**, bukan ditimpa. Karena itu di setiap langkah training kita harus me-*reset* gradien ke nol.
2. **`torch.no_grad()`**: blok di mana operasi tidak dicatat. Dipakai saat meng-update bobot dan saat inferensi/validasi (lebih hemat memori).
3. **`.detach()`**: membuat tensor yang "lepas" dari graph (misalnya untuk plotting atau konversi ke NumPy).

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

for i in range(3):
    y = x**2 + 3 * x
    y.backward()
    print(f"panggilan ke-{i+1}: x.grad = {x.grad.item()}")  # 7, 14, 21 -> terakumulasi!

x.grad.zero_()  # reset gradien
print("setelah zero_():", x.grad.item())

with torch.no_grad():
    z = x * 2
print("z.requires_grad di dalam no_grad:", z.requires_grad)

print("detach:", (x * 2).detach().requires_grad)

## 5. Gradient Descent Manual: Regresi Linear

Sekarang kita gabungkan semuanya. Tujuannya menemukan $w$ dan $b$ agar $\hat{y} = w x + b$ cocok dengan data.

Satu langkah **gradient descent**:
1. **Forward**: hitung prediksi $\hat{y}$.
2. **Loss**: hitung Mean Squared Error.
3. **Backward**: `loss.backward()` untuk mendapatkan gradien.
4. **Update**: $w \leftarrow w - \text{lr} \cdot \frac{\partial L}{\partial w}$ (di dalam `torch.no_grad()`).
5. **Reset gradien** ke nol.

Ini persis yang nanti dilakukan `optimizer.step()` dan `optimizer.zero_grad()` di notebook berikutnya.

### 5.1 Membuat data sintetis

Data dibuat dari garis $y = 2x + 1$ ditambah noise. Model tidak tahu nilai 2 dan 1 ini; ia harus menemukannya sendiri.

In [ ]:
W_ASLI, B_ASLI = 2.0, 1.0
N = 100

X = torch.rand(N, device=device) * 10                        # x di rentang [0, 10)
Y = W_ASLI * X + B_ASLI + torch.randn(N, device=device) * 1.0  # tambah noise

plt.scatter(X.cpu(), Y.cpu(), s=12)
plt.xlabel("x"); plt.ylabel("y"); plt.title("Data sintetis: y = 2x + 1 + noise")
plt.show()

### 5.2 Inisialisasi parameter

Parameter dimulai dari nilai acak kecil dan diberi `requires_grad=True` supaya autograd menghitung gradiennya.

In [ ]:
w = torch.randn(1, device=device, requires_grad=True)
b = torch.zeros(1, device=device, requires_grad=True)
print(f"awal: w = {w.item():.3f}, b = {b.item():.3f}")

### 5.3 Training loop

`LR` (*learning rate*) menentukan besar langkah. Terlalu besar → loss meledak; terlalu kecil → belajar sangat lambat. Coba ubah nilainya nanti!

In [ ]:
LR = 0.01
N_ITER = 500
riwayat_loss = []

for i in range(N_ITER):
    # 1. Forward
    y_pred = w * X + b
    # 2. Loss (MSE)
    loss = ((y_pred - Y) ** 2).mean()
    # 3. Backward: isi w.grad dan b.grad
    loss.backward()
    # 4. Update parameter (tanpa dicatat autograd)
    with torch.no_grad():
        w -= LR * w.grad
        b -= LR * b.grad
    # 5. Reset gradien agar tidak terakumulasi ke iterasi berikutnya
    w.grad.zero_()
    b.grad.zero_()

    riwayat_loss.append(loss.item())
    if i % 100 == 0 or i == N_ITER - 1:
        print(f"iter {i:4d} | loss {loss.item():8.4f} | w {w.item():.3f} | b {b.item():.3f}")

print(f"\nHasil akhir: w = {w.item():.3f} (asli {W_ASLI}), b = {b.item():.3f} (asli {B_ASLI})")

### 5.4 Visualisasi hasil

Kiri: loss turun seiring iterasi (tanda model belajar). Kanan: garis hasil training dibandingkan garis asli.

Catatan: $b$ biasanya konvergen lebih lambat daripada $w$ karena skala $x$ (0–10) membuat gradien $w$ jauh lebih besar. Inilah salah satu alasan **normalisasi input** penting.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(riwayat_loss)
ax[0].set_yscale("log")
ax[0].set_xlabel("iterasi"); ax[0].set_ylabel("MSE (skala log)"); ax[0].set_title("Kurva loss")

xs_garis = torch.linspace(0, 10, 50)
ax[1].scatter(X.cpu(), Y.cpu(), s=12, alpha=0.6, label="data")
ax[1].plot(xs_garis, W_ASLI * xs_garis + B_ASLI, "g--", label="garis asli")
ax[1].plot(xs_garis, w.item() * xs_garis + b.item(), "r", label="hasil training")
ax[1].set_xlabel("x"); ax[1].set_ylabel("y"); ax[1].legend(); ax[1].set_title("Hasil regresi")

plt.tight_layout()
plt.show()

## Ringkasan

- Tensor punya `shape`, `dtype`, dan `device`; pindahkan dengan `.to(device)`.
- `@` untuk perkalian matriks; broadcasting memudahkan penambahan bias.
- `requires_grad=True` + `.backward()` = gradien otomatis di `.grad`.
- Gradien **terakumulasi**, jadi reset setiap langkah; update bobot di dalam `torch.no_grad()`.
- Loop *forward → loss → backward → update → reset* adalah inti semua training deep learning.

## Latihan

Kerjakan sendiri di sel baru di bawah ini.

1. **Learning rate.** Jalankan ulang bagian 5 dengan `LR = 0.001`, `LR = 0.03`, dan `LR = 0.05`. Plot ketiga kurva loss dalam satu grafik. Apa yang terjadi pada LR yang terlalu besar? Jelaskan dengan kata-katamu sendiri mengapa itu terjadi.
2. **Normalisasi input.** Standarisasi `X` menjadi `(X - X.mean()) / X.std()` lalu latih ulang dengan `LR = 0.1`. Bandingkan kecepatan konvergensinya dengan versi asli. Setelah itu, konversi kembali `w` dan `b` hasil training ke skala asli dan cek apakah mendekati 2 dan 1.
3. **Regresi polinomial.** Buat data baru $y = 0.5x^2 - 2x + 3 + \text{noise}$ dengan $x \in [-5, 5]$. Latih model $\hat{y} = a x^2 + b x + c$ memakai gradient descent manual (tiga parameter dengan `requires_grad=True`). Verifikasi juga satu gradien hasil autograd dengan menghitung turunannya secara manual.